In [ ]:
import os
import time
import json
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, accuracy_score

IMG_ROOT = '../data/Images'
label_map = {
    'Negative': 'negative',
    'Neutral':  'neutral',
    'positive': 'positive',
}

## Load image paths and labels

In [ ]:
records = []
for folder, label in label_map.items():
    folder_path = os.path.join(IMG_ROOT, folder)
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            records.append({'path': os.path.join(folder_path, fname), 'label': label})

df = pd.DataFrame(records)

X_train, X_test, y_train, y_test = train_test_split(
    df['path'], df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f'Train: {len(X_train)}  Test: {len(X_test)}')

## Majority class (dummy) classifier

In [ ]:
t0 = time.time()
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_test)
runtime = time.time() - t0

print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(f'Runtime:  {runtime:.2f}s')
print()
print(classification_report(y_test, y_pred, zero_division=0))

report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
meta = {
    'model': 'baseline',
    'accuracy': report['accuracy'],
    'macro_f1': report['macro avg']['f1-score'],
    'negative_f1': report['negative']['f1-score'],
    'neutral_f1': report['neutral']['f1-score'],
    'positive_f1': report['positive']['f1-score'],
    'runtime_seconds': runtime
}
joblib.dump(dummy, '../models/images/fits/baseline.pkl')
with open('../models/images/json/baseline_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved.')